# Dynamic Programming Patterns
### Based on Alvin Zablan's *Learn to Solve Algorithmic Problems & Coding Challenges*

---

## How This Notebook Is Structured

The problems are arranged in **two arcs** that mirror Alvin's progression:

| Arc | Problems | Constraint tightening |
|-----|----------|----------------------|
| **Numeric DP** | `canSum` → `howSum` → `bestSum` | Boolean → any path → shortest path |
| **String DP** | `canConstruct` → `countConstruct` → `allConstruct` | Boolean → count ways → all ways |

Each section follows this structure:
1. **Problem statement** — what are we returning?
2. **Intuition** — how Alvin thinks about it
3. **Recursive tree** — ASCII visualization
4. **Memoization** (Top-Down) — with step-by-step call trace
5. **Tabulation** (Bottom-Up) — with step-by-step table trace
6. **Complexity analysis** — before/after optimization
7. **Gotchas** — the traps Alvin explicitly calls out

---

## Core Idea: Brute Force → Memoization → Tabulation

Every DP problem follows this arc:

```
Brute Force Recursion
       ↓   (cache repeated subproblems)
Memoization  (Top-Down: still recursive, but with a memo dict)
       ↓   (flip the direction, iterate instead)
Tabulation   (Bottom-Up: iterative, build from base cases up)
```

The **memo dict** or **table array** is always indexed by the "remaining problem" — the thing that changes with each recursive call.

In [ ]:
# Utility: pretty call tracer — wraps any function to print indented call/return logs
import functools

_depth = [0]  # mutable int inside list so closures can mutate it

def trace(fn):
    """Decorator that prints indented call/return traces for recursive functions."""
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        indent = '  ' * _depth[0]
        # Only show first two positional args to keep output readable
        display_args = args[:2]
        print(f"{indent}→ {fn.__name__}({', '.join(repr(a) for a in display_args)})")
        _depth[0] += 1
        result = fn(*args, **kwargs)
        _depth[0] -= 1
        print(f"{indent}← returns {repr(result)}")
        return result
    return wrapper

def reset_depth():
    _depth[0] = 0

print("Tracer ready.")

---
# ARC 1: NUMERIC DP
## Problem 1 — `canSum` (Decision Problem)

### Problem Statement
```
canSum(targetSum, numbers) -> bool

Can you generate exactly targetSum by adding numbers from the array?
Numbers can be reused as many times as needed.
```

### Intuition (Alvin's Mental Model)

Think of it as a **tree of remainders**. The root is `targetSum`. Each edge subtracts one number from the array, creating a child node with the new remainder. We're asking: *does any leaf reach exactly 0?*

- Reach `0` → `True` (success — took no more elements, balanced out)
- Go negative → `False` (overshot, dead end)
- If **any** branch returns `True`, the answer is `True` — we can return early

### Recursive Tree — `canSum(7, [3, 5])`
```
                    7
                 /     \
               4         2          ← subtract 3 or 5
             /   \      / \
            1    -1   -1  -3        ← subtract 3 or 5 again
          /   \
         -2   -4                    ← all negative → False

Result: False (7 cannot be made with [3,5])
```

```
canSum(7, [3, 4, 5])

                     7
                 /   |   \
                4    3    2
              / | \  ...
             1  0  ...    ← 0 reached! True bubbles up immediately
                ↑
            SUCCESS
```

In [ ]:
# ── BRUTE FORCE (no memo) ──────────────────────────────────────────────────────
# Time:  O(n^m)  where n = len(numbers), m = targetSum
#                Each level of tree has n branches, tree depth is m
# Space: O(m)    recursion stack depth

def can_sum_brute(target_sum, numbers):
    if target_sum == 0:
        return True
    if target_sum < 0:
        return False
    for num in numbers:
        remainder = target_sum - num
        if can_sum_brute(remainder, numbers):
            return True   # ← early return: first True found, stop exploring
    return False

print(can_sum_brute(7, [3, 4, 5]))   # True  (3+4)
print(can_sum_brute(7, [3, 5]))      # False
print(can_sum_brute(300, [7, 14]))   # False — but SLOW without memo

### Step-by-Step Memoization Trace

**Input:** `canSum(7, [3, 4])` — we trace every call to see memo in action.

```
→ canSum(7, [3,4])            memo={}
  → canSum(4, [3,4])          try subtracting 3
    → canSum(1, [3,4])        try subtracting 3
      → canSum(-2, [3,4])     negative → False
      → canSum(-3, [3,4])     negative → False
      ← memo[1] = False
    → canSum(0, [3,4])        try subtracting 4 → 0! → True
    ← memo[4] = True          early return True
  ← memo[7] = True            early return True (child 4 was True)
```

The memo prevents re-exploring `canSum(4, ...)` if it appears again from a different path.

In [ ]:
# ── MEMOIZATION (Top-Down) ────────────────────────────────────────────────────
# Time:  O(n * m)  — at most m unique subproblems, each tries n numbers
# Space: O(m)      — memo dict has at most m+1 entries + recursion stack depth m

def can_sum_memo(target_sum, numbers, memo=None):
    if memo is None:
        memo = {}                          # fresh memo on each top-level call
    if target_sum in memo:
        return memo[target_sum]            # ← cache hit: skip recomputation
    if target_sum == 0:
        return True
    if target_sum < 0:
        return False

    for num in numbers:
        remainder = target_sum - num
        if can_sum_memo(remainder, numbers, memo):
            memo[target_sum] = True        # cache before returning
            return True                    # early return: found one valid path

    memo[target_sum] = False              # exhausted all paths, cache failure
    return False

# Traced version — run on a small input to see full call tree
@trace
def can_sum_traced(target_sum, numbers, memo=None):
    if memo is None: memo = {}
    if target_sum in memo:
        return memo[target_sum]
    if target_sum == 0: return True
    if target_sum < 0:  return False
    for num in numbers:
        remainder = target_sum - num
        if can_sum_traced(remainder, numbers, memo):
            memo[target_sum] = True
            return True
    memo[target_sum] = False
    return False

reset_depth()
print("=== Trace: canSum(7, [3, 4]) ===")
can_sum_traced(7, [3, 4])

print()
print("=== Correctness checks ===")
print(can_sum_memo(7,   [3, 4, 5]))   # True
print(can_sum_memo(7,   [3, 5]))      # False
print(can_sum_memo(300, [7, 14]))     # False — now fast

### Step-by-Step Tabulation Trace

**Input:** `canSum(7, [3, 4])`

**Strategy:** Build a boolean array of size `targetSum + 1`. Seed index 0 as `True`. Walk left-to-right; wherever we find `True`, mark `i+num` as `True` for each number.

```
Initial:
idx:  0     1     2     3     4     5     6     7
      True  False False False False False False False

i=0 (True): mark 0+3=3 → True, mark 0+4=4 → True
idx:  0     1     2     3     4     5     6     7
      True  False False True  True  False False False

i=1 (False): skip
i=2 (False): skip

i=3 (True): mark 3+3=6 → True, mark 3+4=7 → True ✓
idx:  0     1     2     3     4     5     6     7
      True  False False True  True  False True  True

i=4 (True): mark 4+3=7 → True (already), mark 4+4=8 → out of bounds, skip
...

Return table[7] = True ✓
```

In [ ]:
# ── TABULATION (Bottom-Up) ────────────────────────────────────────────────────
# Time:  O(n * m)  — outer loop m, inner loop n
# Space: O(m)      — table of size m+1

def can_sum_tab(target_sum, numbers):
    table = [False] * (target_sum + 1)
    table[0] = True                        # base case: 0 is always reachable

    for i in range(target_sum + 1):        # iterate every position
        if table[i]:                       # only propagate from reachable positions
            for num in numbers:
                if i + num <= target_sum:  # bounds check
                    table[i + num] = True  # mark reachable

    return table[target_sum]

# Traced tabulation — prints the table state after each propagation step
def can_sum_tab_traced(target_sum, numbers):
    table = [False] * (target_sum + 1)
    table[0] = True
    print(f"Initial: {table}")

    for i in range(target_sum + 1):
        if table[i]:
            for num in numbers:
                if i + num <= target_sum:
                    table[i + num] = True
            print(f"After i={i} (propagated +{numbers}): {table}")

    print(f"Result: table[{target_sum}] = {table[target_sum]}")
    return table[target_sum]

print("=== Trace: canSumTab(7, [3, 4]) ===")
can_sum_tab_traced(7, [3, 4])

print()
print("=== Correctness checks ===")
print(can_sum_tab(7,   [3, 4, 5]))   # True
print(can_sum_tab(7,   [3, 5]))      # False
print(can_sum_tab(300, [7, 14]))     # False — fast

### Complexity Summary — `canSum`

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Brute force | O(n^m) | O(m) | Explodes fast — m=300, n=2 is ~10^90 ops |
| Memoization | O(n·m) | O(m) | At most m unique remainders |
| Tabulation | O(n·m) | O(m) | Same complexity, no recursion stack overhead |

### ⚠️ Gotchas

1. **Early return is valid here** because we only need *one* `True` path. This won't be true for `bestSum`.
2. **Bounds check in tabulation**: `i + num` can exceed `target_sum`. Always guard with `if i + num <= target_sum`.
3. **Memo key is the remainder**, not the original input. Two calls with different `numbers` but same `target_sum` would collide if you shared a memo — always scope memo per top-level call.
4. **Negative numbers in the array** would cause infinite recursion — this problem assumes all numbers are positive.

---
## Problem 2 — `howSum` (Combinatoric Problem)

### Problem Statement
```
howSum(targetSum, numbers) -> list | None

Return ANY combination of numbers that adds to targetSum.
Return None if impossible.
```

### What Changed from `canSum`?
- Return type: `list` (one valid path) instead of `bool`
- On success, we must **bubble the path back up** by appending the chosen number
- Early return still applies — first valid path wins

### Intuition
Same tree as `canSum`. But when a leaf hits `0` (success), instead of returning `True`, it returns `[]`. Each ancestor on the way back up **appends the number it chose** (the edge label) to that list. By the time it bubbles to the root, the list contains the full combination.

### Recursive Tree — `howSum(7, [3, 4])`
```
                 7
               /   \
           [−3]4   [−4]3
           /   \      \
       [−3]1  [−4]0✓  ...
       /   \     ↑
     −2   −3   Returns []
    False False  Parent (4) appends 4 → [4]
                 Parent (7) appends 3 → [4, 3]
                 Final answer: [4, 3]  (means 4+3=7)
```

**Key insight:** The combination is built *on the way back up* the tree, not on the way down.

In [ ]:
# ── MEMOIZATION (Top-Down) ────────────────────────────────────────────────────
# Time:  O(n * m * m)  — m subproblems × n branches × m to copy the result list
# Space: O(m * m)      — memo stores lists of length up to m, for m keys

def how_sum_memo(target_sum, numbers, memo=None):
    if memo is None: memo = {}
    if target_sum in memo:
        return memo[target_sum]       # cache hit
    if target_sum == 0:
        return []                     # success: empty combination
    if target_sum < 0:
        return None                   # overshot

    for num in numbers:
        remainder = target_sum - num
        result = how_sum_memo(remainder, numbers, memo)
        if result is not None:            # child found a valid path
            combination = [*result, num]  # append this edge's number to child's path
            memo[target_sum] = combination
            return combination            # early return: first valid path wins

    memo[target_sum] = None               # all paths exhausted, cache failure
    return None

# Traced version
@trace
def how_sum_traced(target_sum, numbers, memo=None):
    if memo is None: memo = {}
    if target_sum in memo: return memo[target_sum]
    if target_sum == 0:    return []
    if target_sum < 0:     return None
    for num in numbers:
        result = how_sum_traced(target_sum - num, numbers, memo)
        if result is not None:
            combo = [*result, num]
            memo[target_sum] = combo
            return combo
    memo[target_sum] = None
    return None

reset_depth()
print("=== Trace: howSum(7, [3, 4]) ===")
how_sum_traced(7, [3, 4])

print()
print("=== Correctness checks ===")
print(how_sum_memo(7,   [3, 4, 5]))   # [4, 3] or similar
print(how_sum_memo(7,   [3, 5]))      # None
print(how_sum_memo(8,   [2, 3, 5]))   # some combo summing to 8

### Step-by-Step Tabulation Trace

**Input:** `howSum(7, [3, 4])`

Table stores: `None` (unreachable) or a list (the path to reach index `i`)

```
Initial:
idx:  0    1     2     3     4     5     6     7
      []   None  None  None  None  None  None  None

i=0 ([]):  mark table[0+3]=table[3]=[3], table[0+4]=table[4]=[4]
idx:  0    1     2     3     4     5     6     7
      []   None  None  [3]   [4]   None  None  None

i=1,2: None, skip

i=3 ([3]):  mark table[3+3]=table[6]=[3,3], table[3+4]=table[7]=[3,4] ✓
idx:  0    1     2     3     4     5     6     7
      []   None  None  [3]   [4]   None  [3,3] [3,4]

i=4 ([4]):  mark table[4+3]=table[7]=[4,3] (already set, overwrite is ok)
...

Return table[7] = [3,4]  (means 3+4=7) ✓
```

In [ ]:
# ── TABULATION (Bottom-Up) ────────────────────────────────────────────────────
# Time:  O(n * m * m)  — outer m × inner n × list copy m
# Space: O(m * m)      — table of m lists each up to length m

def how_sum_tab(target_sum, numbers):
    table = [None] * (target_sum + 1)
    table[0] = []                              # base case: 0 reachable via empty combo

    for i in range(target_sum + 1):
        if table[i] is not None:               # only propagate from reachable positions
            for num in numbers:
                if i + num <= target_sum:
                    table[i + num] = [*table[i], num]  # copy current path + new num

    return table[target_sum]

def how_sum_tab_traced(target_sum, numbers):
    table = [None] * (target_sum + 1)
    table[0] = []
    print(f"Initial: {table}")

    for i in range(target_sum + 1):
        if table[i] is not None:
            for num in numbers:
                if i + num <= target_sum:
                    table[i + num] = [*table[i], num]
            print(f"After i={i}: {table}")

    print(f"Result: table[{target_sum}] = {table[target_sum]}")
    return table[target_sum]

print("=== Trace: howSumTab(7, [3, 4]) ===")
how_sum_tab_traced(7, [3, 4])

print()
print("=== Correctness checks ===")
print(how_sum_tab(7,   [3, 4, 5]))
print(how_sum_tab(7,   [3, 5]))
print(how_sum_tab(8,   [2, 3, 5]))

### Complexity Summary — `howSum`

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Brute force | O(n^m · m) | O(m²) | Extra m factor from list copying |
| Memoization | O(n·m²) | O(m²) | m unique remainders, each stores list of len ≤ m |
| Tabulation | O(n·m²) | O(m²) | Same — list copies dominate |

### ⚠️ Gotchas

1. **Returning `[]` vs `None`**: `[]` is the base case (successfully reached 0). `None` is failure. Don't confuse them with `if result:` — an empty list is falsy in Python. Always use `if result is not None:`.
2. **Always copy the list** with `[*result, num]` — never mutate the returned result in place. Memo stores references; mutating them corrupts cached values.
3. **Order in the list**: The number is appended *after* the child's list (building from leaves up). So the combination reads in reverse order of how numbers were chosen going down the tree.

---
## Problem 3 — `bestSum` (Optimization Problem)

### Problem Statement
```
bestSum(targetSum, numbers) -> list | None

Return the SHORTEST combination of numbers that adds to targetSum.
Return None if impossible.
```

### What Changed from `howSum`?
- We want the **shortest** combination, not just *any*
- **Cannot return early** — we must explore ALL branches
- Need to track a `shortest` variable and update it only when we find a strictly shorter path

### Intuition

The tree structure is identical to `howSum`. But instead of stopping at the first `True` leaf, we must visit **every leaf**, compare lengths, and keep only the shortest. Alvin calls this the key insight: *removing early return forces exhaustive search*.

```
bestSum(7, [3, 4, 7])

                        7
                   /    |    \
                  4     3     0✓ → []  (len 0 → combo [7])
                /   \   ...
               1     0✓ → []         (len 0 → combo [4,3])
             /   \
           −2   −3

Paths found:
  [7]     → length 1  ← shortest!
  [4, 3]  → length 2
  [3, 4]  → length 2
  [3,3,?] → never reaches 7

Return [7]
```

In [ ]:
# ── MEMOIZATION (Top-Down) ────────────────────────────────────────────────────
# Time:  O(n * m * m)  — same as howSum; exhaustive search doesn't change big-O
# Space: O(m * m)

def best_sum_memo(target_sum, numbers, memo=None):
    if memo is None: memo = {}
    if target_sum in memo: return memo[target_sum]
    if target_sum == 0:    return []           # success: empty combo
    if target_sum < 0:     return None         # overshot

    shortest = None                            # track the best so far

    for num in numbers:
        remainder = target_sum - num
        result = best_sum_memo(remainder, numbers, memo)

        if result is not None:                 # found a valid path
            combination = [*result, num]
            # Update only if this is the first success OR strictly shorter
            if shortest is None or len(combination) < len(shortest):
                shortest = combination         # NO early return here!

    memo[target_sum] = shortest                # cache best for this remainder
    return shortest

@trace
def best_sum_traced(target_sum, numbers, memo=None):
    if memo is None: memo = {}
    if target_sum in memo: return memo[target_sum]
    if target_sum == 0:    return []
    if target_sum < 0:     return None
    shortest = None
    for num in numbers:
        result = best_sum_traced(target_sum - num, numbers, memo)
        if result is not None:
            combo = [*result, num]
            if shortest is None or len(combo) < len(shortest):
                shortest = combo
    memo[target_sum] = shortest
    return shortest

reset_depth()
print("=== Trace: bestSum(7, [3, 4, 7]) ===")
best_sum_traced(7, [3, 4, 7])

print()
print("=== Correctness checks ===")
print(best_sum_memo(7,   [3, 4, 7]))      # [7]
print(best_sum_memo(8,   [2, 3, 5]))      # [3, 5] or [5, 3]
print(best_sum_memo(100, [1, 2, 5, 25]))  # [25, 25, 25, 25]

### Step-by-Step Tabulation Trace

**Input:** `bestSum(7, [3, 4, 7])`

At each position, only replace the stored combination if the new one is **strictly shorter**.

```
Initial:
idx:  0    1     2     3     4     5     6     7
      []   None  None  None  None  None  None  None

i=0 ([]): propagate +3 → [3], +4 → [4], +7 → [7]
idx:  0    1     2     3     4     5     6     7
      []   None  None  [3]   [4]   None  None  [7]

i=3 ([3]): propagate +3 → [3,3], +4 → [3,4], +7 → out of bounds
  table[6]: None → [3,3]  (len 2)
  table[7]: [7] (len 1) vs [3,4] (len 2) → keep [7]  ← shorter wins!

i=4 ([4]): propagate +3 → [4,3], +4 → out of bounds
  table[7]: [7] (len 1) vs [4,3] (len 2) → keep [7]

...

Return table[7] = [7] ✓
```

In [ ]:
# ── TABULATION (Bottom-Up) ────────────────────────────────────────────────────
# Time:  O(n * m * m)
# Space: O(m * m)

def best_sum_tab(target_sum, numbers):
    table = [None] * (target_sum + 1)
    table[0] = []

    for i in range(target_sum + 1):
        if table[i] is not None:
            for num in numbers:
                if i + num <= target_sum:
                    candidate = [*table[i], num]
                    future = table[i + num]
                    # Update only if future is empty OR candidate is shorter
                    if future is None or len(candidate) < len(future):
                        table[i + num] = candidate

    return table[target_sum]

def best_sum_tab_traced(target_sum, numbers):
    table = [None] * (target_sum + 1)
    table[0] = []
    print(f"Initial: {table}")

    for i in range(target_sum + 1):
        if table[i] is not None:
            for num in numbers:
                if i + num <= target_sum:
                    candidate = [*table[i], num]
                    future = table[i + num]
                    if future is None or len(candidate) < len(future):
                        table[i + num] = candidate
            print(f"After i={i}: {table}")

    print(f"Result: table[{target_sum}] = {table[target_sum]}")
    return table[target_sum]

print("=== Trace: bestSumTab(7, [3, 4, 7]) ===")
best_sum_tab_traced(7, [3, 4, 7])

print()
print("=== Correctness checks ===")
print(best_sum_tab(7,   [3, 4, 7]))
print(best_sum_tab(8,   [2, 3, 5]))
print(best_sum_tab(100, [1, 2, 5, 25]))

### Complexity Summary — `bestSum`

| Approach | Time | Space |
|----------|------|-------|
| Brute force | O(n^m · m) | O(m²) |
| Memoization | O(n·m²) | O(m²) |
| Tabulation | O(n·m²) | O(m²) |

### ⚠️ Gotchas

1. **No early return** — this is the critical difference from `howSum`. If you return early, you might return a longer combination and miss the optimal one.
2. **`shortest = None` as sentinel** — don't initialize to `[]` (that would be the real base case value for remainder=0).
3. **Tabulation update condition**: `if future is None or len(candidate) < len(future)` — not `<=`. Using `<` keeps the *first* shortest found; `<=` would keep overwriting with any equal-length path.
4. **The three canSum/howSum/bestSum problems form a template**: Decision (bool) → Any Solution (list) → Optimal Solution (shortest list). Recognize this pattern on new problems.

---
# ARC 2: STRING DP
## Problem 4 — `canConstruct` (String Decision Problem)

### Problem Statement
```
canConstruct(target, wordBank) -> bool

Return True if 'target' can be built by concatenating words from wordBank.
Words can be reused. Order matters.
```

### The Parallel to `canSum`

| Numeric | String |
|---------|--------|
| `targetSum` | `target` string |
| Subtract a number | Remove a prefix word |
| Remainder (int) | Suffix (string) |
| Base case: `0` | Base case: `''` |
| Base case: `< 0` | Base case: word not prefix |

### Intuition — The Prefix Rule

Alvin's critical warning: **you cannot pull words from the middle of the string**. If `target = "abcdef"` and word = `"bcd"`, you might be tempted to match it at index 1. But that leaves `"a"` dangling on the left — invalid, because words must be concatenated from the beginning.

**Rule:** A word can only be used if it matches the *start* (prefix) of the current target. If it matches, slice it off and recurse on the suffix.

### Recursive Tree — `canConstruct("abcdef", ["ab", "abc", "cd", "def", "abcd"])`
```
"abcdef"
├─[ab]─ "cdef"
│        └─[cd]─ "ef"   → no prefix matches → False
│
├─[abc]─ "def"
│        └─[def]─ ""    → True ✓
│
└─[abcd]─ "ef"  → no prefix matches → False

Result: True (via "abc" + "def")
```

In [ ]:
# ── MEMOIZATION (Top-Down) ────────────────────────────────────────────────────
# Time:  O(n * m²)   — m unique suffixes × n words × m for slicing
# Space: O(m²)       — memo stores up to m string keys of total length O(m²)
#   where m = len(target), n = len(wordBank)

def can_construct_memo(target, word_bank, memo=None):
    if memo is None: memo = {}
    if target in memo:  return memo[target]
    if target == '':    return True           # empty string: success

    for word in word_bank:
        # Critical: word must be a PREFIX — starts at index 0
        if target.startswith(word):
            suffix = target[len(word):]       # slice off the matched prefix
            if can_construct_memo(suffix, word_bank, memo):
                memo[target] = True
                return True                   # early return: any valid path wins

    memo[target] = False
    return False

@trace
def can_construct_traced(target, word_bank, memo=None):
    if memo is None: memo = {}
    if target in memo: return memo[target]
    if target == '': return True
    for word in word_bank:
        if target.startswith(word):
            suffix = target[len(word):]
            if can_construct_traced(suffix, word_bank, memo):
                memo[target] = True
                return True
    memo[target] = False
    return False

reset_depth()
print("=== Trace: canConstruct('abcdef', ['ab','abc','cd','def','abcd']) ===")
can_construct_traced('abcdef', ['ab', 'abc', 'cd', 'def', 'abcd'])

print()
print("=== Correctness checks ===")
print(can_construct_memo('abcdef',     ['ab', 'abc', 'cd', 'def', 'abcd']))  # True
print(can_construct_memo('skateboard', ['bo', 'rd', 'ate', 't', 'ska', 'sk', 'boar']))  # False
print(can_construct_memo('eeeeeeeeeeeeeeef', ['e', 'ee', 'eee', 'eeee']))  # False — fast with memo

### Step-by-Step Tabulation Trace

**Input:** `canConstruct("abcd", ["ab", "cd", "abc", "d"])`

Table is indexed by **position** in the string (0 = start, `len(target)` = end).

```
target = "abcd"  (positions 0..4)

Initial:
pos:   0     1     2     3     4
       True  False False False False

i=0 (True):  
  "ab"  matches target[0:2]="ab"  → mark pos 2 True
  "abc" matches target[0:3]="abc" → mark pos 3 True

pos:   0     1     2     3     4
       True  False True  True  False

i=1 (False): skip

i=2 (True):
  "cd" matches target[2:4]="cd"  → mark pos 4 True ✓

pos:   0     1     2     3     4
       True  False True  True  True

i=3 (True):
  "d" matches target[3:4]="d"   → pos 4 already True

Return table[4] = True ✓
```

In [ ]:
# ── TABULATION (Bottom-Up) ────────────────────────────────────────────────────
# Table is indexed by position in the string, not by a remainder int.
# Table size: len(target) + 1  (to include the empty-string position at the end)

def can_construct_tab(target, word_bank):
    m = len(target)
    table = [False] * (m + 1)
    table[0] = True                            # empty string is always constructible

    for i in range(m + 1):
        if table[i]:
            for word in word_bank:
                end = i + len(word)
                # Check if 'word' matches the substring starting at position i
                if end <= m and target[i:end] == word:
                    table[end] = True

    return table[m]

def can_construct_tab_traced(target, word_bank):
    m = len(target)
    table = [False] * (m + 1)
    table[0] = True
    print(f"Initial: {table}  (positions 0..{m} of '{target}')")

    for i in range(m + 1):
        if table[i]:
            for word in word_bank:
                end = i + len(word)
                if end <= m and target[i:end] == word:
                    table[end] = True
            print(f"After i={i} ('{target[i:]}' remaining): {table}")

    print(f"Result: table[{m}] = {table[m]}")
    return table[m]

print("=== Trace: canConstructTab('abcd', ['ab','cd','abc','d']) ===")
can_construct_tab_traced('abcd', ['ab', 'cd', 'abc', 'd'])

print()
print("=== Correctness checks ===")
print(can_construct_tab('abcdef',     ['ab', 'abc', 'cd', 'def', 'abcd']))  # True
print(can_construct_tab('skateboard', ['bo', 'rd', 'ate', 't', 'ska', 'sk', 'boar']))  # False

### Complexity Summary — `canConstruct`

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Brute force | O(n^m · m) | O(m²) | m = target length, n = word bank size |
| Memoization | O(n·m²) | O(m²) | m for slicing per call |
| Tabulation | O(n·m²) | O(m) | Table only stores booleans |

### ⚠️ Gotchas

1. **Prefix-only matching** — the most important rule. Using `word in target` would incorrectly match words in the middle.
2. **The memo key is the suffix string**, not an index. This is fine but means the memo dict has string keys. The tabulation approach uses integer indices, which is slightly more efficient.
3. **String slicing is O(k)** where k is the length of the slice — this adds the extra `m` factor to complexity.
4. **Table index = position in string**, not a value. This is a conceptual shift from numeric DP where the index *was* the value.

---
## Problem 5 — `countConstruct` (String Count Problem)

### Problem Statement
```
countConstruct(target, wordBank) -> int

Return the NUMBER of ways target can be constructed from wordBank.
Words can be reused.
```

### What Changed from `canConstruct`?
- Return type: `int` instead of `bool`
- Cannot return early — must count **all** valid paths
- Accumulate counts from all branches instead of short-circuiting
- Directly mirrors `howSum`→`bestSum` escalation but for strings

### Intuition

Same prefix tree as `canConstruct`. But instead of returning `True` the moment we find one valid path, we sum up the counts from every valid branch. Base case `''` returns `1` (one way: the empty combination). Dead ends return `0`.

### Recursive Tree — `countConstruct("purple", ["purp", "p", "ur", "le", "purpl"])`
```
"purple"
├─[purp]─ "le"
│          └─[le]─ "" → 1
│          count: 1
│
└─[p]─ "urple"
        └─[ur]─ "ple"
                 └─ (no match for 'ple') → 0
        count: 0

Total: 1 + 0 = 1
```

In [ ]:
# ── MEMOIZATION (Top-Down) ────────────────────────────────────────────────────
# Time:  O(n * m²)
# Space: O(m²)

def count_construct_memo(target, word_bank, memo=None):
    if memo is None: memo = {}
    if target in memo: return memo[target]
    if target == '':   return 1              # one way to make empty string

    total_count = 0                          # accumulate: no early return!

    for word in word_bank:
        if target.startswith(word):
            suffix = target[len(word):]
            count_for_rest = count_construct_memo(suffix, word_bank, memo)
            total_count += count_for_rest    # add child's count to running total

    memo[target] = total_count
    return total_count

@trace
def count_construct_traced(target, word_bank, memo=None):
    if memo is None: memo = {}
    if target in memo: return memo[target]
    if target == '': return 1
    total = 0
    for word in word_bank:
        if target.startswith(word):
            total += count_construct_traced(target[len(word):], word_bank, memo)
    memo[target] = total
    return total

reset_depth()
print("=== Trace: countConstruct('purple', ['purp','p','ur','le','purpl']) ===")
count_construct_traced('purple', ['purp', 'p', 'ur', 'le', 'purpl'])

print()
print("=== Correctness checks ===")
print(count_construct_memo('purple',       ['purp', 'p', 'ur', 'le', 'purpl']))  # 2
print(count_construct_memo('abcdef',       ['ab', 'abc', 'cd', 'def', 'abcd', 'ef']))  # 4
print(count_construct_memo('skateboard',   ['bo', 'rd', 'ate', 't', 'ska', 'sk', 'boar']))  # 0
print(count_construct_memo('eeeeeeeeeeef', ['e', 'ee', 'eee', 'eeee']))  # 0 — fast

### Step-by-Step Tabulation Trace

**Input:** `countConstruct("abcd", ["ab", "cd", "abc", "d"])`

Table stores counts (int). Instead of setting `True`, we **add** the count at position `i` into the count at `i + len(word)`.

```
Initial:
pos:  0   1   2   3   4
      1   0   0   0   0

i=0 (count=1):
  "ab"  matches → table[2] += 1  → table[2] = 1
  "abc" matches → table[3] += 1  → table[3] = 1

pos:  0   1   2   3   4
      1   0   1   1   0

i=1 (count=0): nothing to propagate

i=2 (count=1):
  "cd" matches target[2:4]="cd" → table[4] += 1  → table[4] = 1

pos:  0   1   2   3   4
      1   0   1   1   1

i=3 (count=1):
  "d" matches target[3:4]="d" → table[4] += 1  → table[4] = 2

Return table[4] = 2  ("ab"+"cd" and "abc"+"d") ✓
```

In [ ]:
# ── TABULATION (Bottom-Up) ────────────────────────────────────────────────────
# Time:  O(n * m²)
# Space: O(m)     — table stores ints, not lists

def count_construct_tab(target, word_bank):
    m = len(target)
    table = [0] * (m + 1)
    table[0] = 1                             # one way to construct empty string

    for i in range(m + 1):
        if table[i] > 0:                     # only propagate from reachable positions
            for word in word_bank:
                end = i + len(word)
                if end <= m and target[i:end] == word:
                    table[end] += table[i]   # accumulate: add ways, don't overwrite

    return table[m]

def count_construct_tab_traced(target, word_bank):
    m = len(target)
    table = [0] * (m + 1)
    table[0] = 1
    print(f"Initial: {table}")

    for i in range(m + 1):
        if table[i] > 0:
            for word in word_bank:
                end = i + len(word)
                if end <= m and target[i:end] == word:
                    table[end] += table[i]
            print(f"After i={i}: {table}")

    print(f"Result: table[{m}] = {table[m]}")
    return table[m]

print("=== Trace: countConstructTab('abcd', ['ab','cd','abc','d']) ===")
count_construct_tab_traced('abcd', ['ab', 'cd', 'abc', 'd'])

print()
print("=== Correctness checks ===")
print(count_construct_tab('purple',     ['purp', 'p', 'ur', 'le', 'purpl']))  # 2
print(count_construct_tab('abcdef',     ['ab', 'abc', 'cd', 'def', 'abcd', 'ef']))  # 4
print(count_construct_tab('skateboard', ['bo', 'rd', 'ate', 't', 'ska', 'sk', 'boar']))  # 0

### Complexity Summary — `countConstruct`

| Approach | Time | Space |
|----------|------|-------|
| Brute force | O(n^m · m) | O(m²) |
| Memoization | O(n·m²) | O(m²) |
| Tabulation | O(n·m²) | O(m) | Stores ints, not lists |

### ⚠️ Gotchas

1. **Accumulate, don't assign** — `table[end] += table[i]`, not `= table[i]`. Multiple paths can reach the same position; we must sum them all.
2. **No early return** — same rule as `bestSum`. We need all paths, not just the first.
3. **Base case is `1` not `True`** — the empty string can be constructed in exactly 1 way (by taking nothing). Every other position starts at `0`.
4. **Propagate by current count**: if `table[i] = 3`, then all positions reachable from `i` each get `+3`. This is the multiplier effect of DP counting.

---
## Problem 6 — `allConstruct` (String Exhaustive Problem)

### Problem Statement
```
allConstruct(target, wordBank) -> list[list[str]]

Return ALL ways target can be constructed from wordBank as a 2D array.
Each sub-array is one valid combination of words.
Return [[]] if target is empty. Return [] if impossible.
```

### What Changed from `countConstruct`?
- Return type: `list[list[str]]` (all paths explicitly) instead of an int count
- **No memoization optimization possible** — we must store all combinations, not just a count. The memo doesn't actually save work here in the worst case.
- Both approaches are O(n^m) in the worst case — this is inherent: there can be exponentially many valid combinations

### Intuition

Same prefix tree. When a child returns its 2D array of ways, the parent **prepends its chosen word** to every sub-array in that 2D result. All these new sub-arrays get collected into a master result.

### Base Case Special Attention
```
allConstruct('', wordBank) → [[]]   # one way: the empty combination
allConstruct('xyz', [])    → []     # no ways at all
```
The distinction between `[[]]` (one empty combination) and `[]` (no combinations) is critical — confusing them breaks the entire recursive chain.

### Recursive Tree — `allConstruct("abcd", ["ab", "cd", "abc", "d"])`
```
"abcd"
├─[ab]── "cd"
│         └─[cd]── "" → [[]] → map: [["cd"]] → parent: [["ab","cd"]]
│
└─[abc]─ "d"
          └─[d]── "" → [[]] → map: [["d"]] → parent: [["abc","d"]]

Result: [["ab","cd"], ["abc","d"]]
```

In [ ]:
# ── MEMOIZATION (Top-Down) ────────────────────────────────────────────────────
# Time:  O(n^m)   — worst case: exponentially many combinations
# Space: O(n^m)   — must store all combinations
# NOTE: Memoization doesn't give an asymptotic speedup here because the total
#       output size is inherently exponential. But it still helps avoid
#       redundant suffix recomputation in practice.

def all_construct_memo(target, word_bank, memo=None):
    if memo is None: memo = {}
    if target in memo: return memo[target]
    if target == '':   return [[]]           # one way: the empty combination

    all_ways = []                            # collect ALL valid paths

    for word in word_bank:
        if target.startswith(word):
            suffix = target[len(word):]
            suffix_ways = all_construct_memo(suffix, word_bank, memo)

            # Prepend current word to each of the child's combinations
            target_ways = [[word, *way] for way in suffix_ways]
            all_ways.extend(target_ways)     # collect into master list

    memo[target] = all_ways
    return all_ways

@trace
def all_construct_traced(target, word_bank, memo=None):
    if memo is None: memo = {}
    if target in memo: return memo[target]
    if target == '': return [[]]
    all_ways = []
    for word in word_bank:
        if target.startswith(word):
            suffix_ways = all_construct_traced(target[len(word):], word_bank, memo)
            all_ways.extend([[word, *way] for way in suffix_ways])
    memo[target] = all_ways
    return all_ways

reset_depth()
print("=== Trace: allConstruct('abcd', ['ab','cd','abc','d']) ===")
all_construct_traced('abcd', ['ab', 'cd', 'abc', 'd'])

print()
print("=== Correctness checks ===")
print(all_construct_memo('abcdef',     ['ab', 'abc', 'cd', 'def', 'abcd', 'ef']))
print(all_construct_memo('purple',     ['purp', 'p', 'ur', 'le', 'purpl']))
print(all_construct_memo('skateboard', ['bo', 'rd', 'ate', 't', 'ska', 'sk', 'boar']))  # []

### Step-by-Step Tabulation Trace

**Input:** `allConstruct("abcd", ["ab", "cd", "abc", "d"])`

Table stores a list of combinations at each position. We **push** new combinations forward.

```
Initial:
pos:  0      1    2    3    4
      [[]]   []   []   []   []

i=0 ([[]]):
  "ab"  matches → table[2].push([[]+["ab"]]) → table[2]=[['ab']]
  "abc" matches → table[3].push([[]+["abc"]]) → table[3]=[['abc']]

pos:  0      1    2         3          4
      [[]]   []   [['ab']]  [['abc']]  []

i=2 ([['ab']]):
  "cd" matches target[2:4] → table[4].push(['ab']+['cd']) → table[4]=[['ab','cd']]

pos:  0      1    2         3          4
      [[]]   []   [['ab']]  [['abc']]  [['ab','cd']]

i=3 ([['abc']]):
  "d" matches target[3:4] → table[4].push(['abc']+['d']) → table[4]=[['ab','cd'],['abc','d']]

Return table[4] = [['ab','cd'], ['abc','d']] ✓
```

In [ ]:
# ── TABULATION (Bottom-Up) ────────────────────────────────────────────────────
# Time:  O(n^m)   — inherent: exponential output size
# Space: O(n^m)

def all_construct_tab(target, word_bank):
    m = len(target)
    # Each position holds a list of combinations (2D→3D structure)
    table = [[] for _ in range(m + 1)]      # unique empty lists per position!
    table[0] = [[]]                          # one way to reach position 0: empty combo

    for i in range(m + 1):
        if table[i]:                         # only propagate from reachable positions
            for word in word_bank:
                end = i + len(word)
                if end <= m and target[i:end] == word:
                    # Append 'word' to each combination at position i
                    new_combos = [combo + [word] for combo in table[i]]
                    table[end].extend(new_combos)  # push forward

    return table[m]

def all_construct_tab_traced(target, word_bank):
    m = len(target)
    table = [[] for _ in range(m + 1)]
    table[0] = [[]]
    print(f"Initial: {table}")

    for i in range(m + 1):
        if table[i]:
            for word in word_bank:
                end = i + len(word)
                if end <= m and target[i:end] == word:
                    new_combos = [combo + [word] for combo in table[i]]
                    table[end].extend(new_combos)
            print(f"After i={i}: {table}")

    print(f"Result: table[{m}] = {table[m]}")
    return table[m]

print("=== Trace: allConstructTab('abcd', ['ab','cd','abc','d']) ===")
all_construct_tab_traced('abcd', ['ab', 'cd', 'abc', 'd'])

print()
print("=== Correctness checks ===")
print(all_construct_tab('purple',     ['purp', 'p', 'ur', 'le', 'purpl']))
print(all_construct_tab('abcdef',     ['ab', 'abc', 'cd', 'def', 'abcd', 'ef']))
print(all_construct_tab('skateboard', ['bo', 'rd', 'ate', 't', 'ska', 'sk', 'boar']))

### Complexity Summary — `allConstruct`

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Brute force | O(n^m) | O(n^m) | Inherent: output can be exponentially large |
| Memoization | O(n^m) | O(n^m) | Memo helps avoid re-exploring suffixes but output size unchanged |
| Tabulation | O(n^m) | O(n^m) | Same — complexity is dominated by output |

### ⚠️ Gotchas

1. **`[[]]` vs `[]` base cases** — `[[]]` means "one valid way (the empty combination)". `[]` means "zero valid ways". This distinction propagates through the entire recursion. Returning `[]` from the base case would cause all valid combinations to be silently dropped.
2. **Always copy lists** — `[combo + [word] for combo in table[i]]` creates new lists. Never mutate `combo` in place — multiple future positions may share references to the same list.
3. **Tabulation needs `[[] for _ in range(m+1)]`**, not `[[]] * (m+1)`. The latter creates m+1 references to the *same* list — mutating one mutates all.
4. **Memoization gives limited benefit here**: unlike previous problems where memo hits avoid recomputation, here we must materialize all combinations anyway. The complexity cannot be reduced below the output size.
5. **No early return is possible or desirable** — by definition, we want every path.

---
# Summary: The DP Pattern Taxonomy

## The Two Axes

Every problem in this notebook sits on two axes:

```
                  NUMERIC DP          STRING DP
                  (subtract num)      (strip prefix)
                  ──────────────────────────────────
Decision?    →    canSum              canConstruct
Any path?    →    howSum              (howConstruct — not in video)
Optimal?     →    bestSum             (bestConstruct — not in video)
Count paths? →    (countSum)          countConstruct
All paths?   →    (allSum)            allConstruct
```

## The Recipe

| Step | What to decide |
|------|----------------|
| 1 | What is my **subproblem**? (What shrinks each recursion?) |
| 2 | What are my **base cases**? (When do I stop?) |
| 3 | What am I **returning**? (bool / any-list / shortest-list / count / all-lists) |
| 4 | Can I **return early**, or must I **exhaust all branches**? |
| 5 | What is my **memo key**? (The subproblem identifier) |
| 6 | For tabulation: what is my **seed** and how do I **propagate**? |

## Memoization vs Tabulation — When to Use Which

| | Memoization | Tabulation |
|--|-------------|------------|
| Direction | Top-Down (recursive) | Bottom-Up (iterative) |
| Best when | Subproblem space is sparse | All subproblems are needed |
| Risk | Stack overflow on deep recursion | Harder to visualize initially |
| Overhead | Function call stack | None |
| Code style | Closer to brute force (easier to derive) | Requires inverting your thinking |